# 06 — Prophet (optional)

If `prophet` is not installed, the wrapper logs a warning and returns a naive fallback with `status=unavailable`. See `docs/forecasting_methodology.md`.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.utils.logging_config import setup_logging
from src.utils.helpers import load_config, load_model_config, resolve_path
setup_logging("INFO")
CONFIG = load_config()
print("Independent M5-schema project. DATA_DIR =", resolve_path(CONFIG["paths"]["data_dir"]))


In [ ]:
import pandas as pd
from src.forecasting.prophet_model import PROPHET_AVAILABLE, fit_predict_prophet
from src.forecasting.evaluation import chronological_split

print("Prophet available:", PROPHET_AVAILABLE)
fact = pd.read_parquet(resolve_path(CONFIG["paths"]["processed_dir"]) / "fact_daily_sales.parquet")
item, store = fact.groupby(["item_id", "store_id"])["revenue"].sum().idxmax()
series = fact[(fact.item_id == item) & (fact.store_id == store)].sort_values("date")
train, valid = chronological_split(series, "date", 28)
pred = fit_predict_prophet(train, pd.to_datetime(valid["date"]), load_model_config())
pred.head()
